# GAVAGAI on Kaggle (T4)

Referential alignment for infant-scale vision-language models &mdash; **BabyVLM Workshop @ NeurIPS 2026**.

**Before running:** Settings &rarr; Accelerator &rarr; **GPU T4 x2**, and Internet **ON**.

Total runtime for everything below is roughly 4-6 GPU-hours, well inside the 30 h/week quota.
Section 5 is the only expensive part; sections 1-4 take minutes.


## 0. Setup


In [ ]:
!git clone -q https://github.com/kylian07/baby-vlm.git /kaggle/working/gavagai || true
%cd /kaggle/working/gavagai
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, '/kaggle/working/gavagai')
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.device_count(), 'GPU(s)')


In [ ]:
# Every proposition in METHOD.md has an executable check. Run them first.
!python -m pytest tests/ -q


## 1. Benchmark audit &mdash; are these tasks measuring grounding?

Answer each DevCV multiple-choice item with a 32x32 luminance + RGB-histogram descriptor:
no training, no network, and **no access to the prompt**.

On the bundled website samples this solves Left/Right 24/24. Here we regenerate it on the
full public Ego4D release, which is the number worth quoting.


In [ ]:
!pip install -q huggingface_hub
!hf download wsashawn/devcv_toolbox_ego4d --repo-type dataset --local-dir data/Ego4D 2>&1 | tail -2


In [ ]:
!python scripts/text_blind_audit.py --roots data/Ego4D


## 2. Controlled experiments (CPU, minutes)

The headline simulation: an autoregressive captioning objective &mdash; the one BabyVLM-V2
actually uses &mdash; with and without the referential-alignment term, swept across
referential ambiguity. Readout is picture-vocabulary accuracy on **held-out exemplars**.


In [ ]:
!python scripts/run_simulation.py ar --seeds 5 --steps 600 --corpus-size 1500


### 2b. Scope condition: where the method does *not* help

In a **contrastive** setting the exclusivity constraint gives no reliable benefit, because a
batch-level InfoNCE already draws negatives from the marginal and suppresses hub collapse.
We report this rather than hide it &mdash; it is why the method targets autoregressive models.


In [ ]:
!python scripts/run_simulation.py efficiency --seeds 5
!python scripts/run_simulation.py rho --seeds 5


## 3. Psychometric check &mdash; Yu & Smith (2007)

An unconstrained ideal observer is at ceiling on the original design, so the honest question
is what capacity limit reproduces the human ordering (0.889 / 0.778 / 0.556) with a single
shared free parameter.


In [ ]:
!python scripts/run_simulation.py yusmith --seeds 20


## 4. Figures


In [ ]:
!python scripts/make_figures.py
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('docs/figures').glob('*.png')): print(p); display(Image(str(p)))


## 5. Real training run (GPU)

Two runs differing by exactly one loss term, on identical data, parameters and readout.

`--data synthetic` needs no downloads and has known ground truth; swap in any
`{"image", "text"}` json/jsonl (COCO, Localized Narratives, SAYCam) with `--data` and
`--image-root`. Each run checkpoints so it survives the 12 h session limit &mdash;
resume with `--resume runs/<name>/ckpt.pt`.


In [ ]:
!python scripts/train.py --data synthetic --synthetic-n 20000 \
    --steps 6000 --batch 128 --aux-weight 0.0 \
    --eval-every 500 --out runs/ar_baseline --seed 0


In [ ]:
!python scripts/train.py --data synthetic --synthetic-n 20000 \
    --steps 6000 --batch 128 --aux-weight 1.0 --rho 1.0 --use-null \
    --eval-every 500 --out runs/gavagai --seed 0


In [ ]:
# Naive alignment: no null bin, row-softmax E-step. This is the ablation that
# should be *worse* than the baseline when most speech is non-referential.
!python scripts/train.py --data synthetic --synthetic-n 20000 \
    --steps 6000 --batch 128 --aux-weight 1.0 --rho 0.0 \
    --eval-every 500 --out runs/naive_align --seed 0


In [ ]:
!python scripts/make_figures.py
from IPython.display import Image, display
display(Image('docs/figures/training.png'))


## 6. Collect results

Everything lands in `results/` and `runs/`; download them from the Kaggle output pane.


In [ ]:
import json, pathlib
for p in sorted(pathlib.Path('results').glob('*.json')) + sorted(pathlib.Path('runs').glob('*/history.json')):
    print('=' * 70); print(p)
    print(json.dumps(json.loads(p.read_text()), indent=2)[:1500])
